# Notebook 02 — Content-Based Similar-Movie Lists (WBS 3.3)

Input: `curated_movies` (genres) trên Drive. Output: `similar_movies.json` (CONTRACTS §3.2, Top-M = 20).

**Domain Guard (PLAN §8):** 87,585 phim ⟹ similarity matrix đầy 87,585² = 7.66 tỷ cells —
KHÔNG thể materialize. Dùng **block-matrix numpy** (float32, block 2048 hàng × 87,585 cột
= ~717MB/block, chạy 43 blocks) — mỗi block chỉ giữ top-M = 20 bằng argpartition.
Genres multi-hot là core signal; TF-IDF tags OPTIONAL (EDA §9: tag coverage thưa).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/movielens32m'
CURATED = f'{BASE}/curated'; ARTIFACTS = f'{BASE}/artifacts'; EVID = f'{BASE}/evidence'
import os
for d in [ARTIFACTS, EVID]: os.makedirs(d, exist_ok=True)

!pip install -q pyspark==3.5.7
from pyspark.sql import SparkSession, functions as F
import os, shutil
if not os.path.isdir('/content/curated/curated_movies'):
    shutil.copytree(CURATED, '/content/curated')
spark = SparkSession.builder.master('local[*]').appName('content-based').config('spark.driver.memory', '6g').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
movies = spark.read.parquet(f'/content/curated/curated_movies')
n_movies = movies.count()
print(f'movies: {n_movies:,}')

In [ ]:
# Genres multi-hot matrix (numpy float32) — dùng Spark chỉ để fetch, tính toán numpy block
import numpy as np, pandas as pd
mrows = movies.select('movieId', 'genre_list').toPandas()
mrows['genre_list'] = mrows['genre_list'].apply(lambda g: [x for x in g if x != '(no genres listed)'])
genres_vocab = sorted({g for lst in mrows['genre_list'] for g in lst})
g_index = {g: i for i, g in enumerate(genres_vocab)}
D = len(genres_vocab)
print(f'{D} genres: {genres_vocab}')

ids = mrows['movieId'].values
X = np.zeros((len(mrows), D), dtype=np.float32)
for r, lst in enumerate(mrows['genre_list']):
    for g in lst: X[r, g_index[g]] = 1.0
norms = np.linalg.norm(X, axis=1); norms[norms == 0] = 1e-9   # no-genres movies → similarity ~ 0
Xn = X / norms[:, None]
print(f'X: {X.shape}, nnz={np.count_nonzero(X):,}')

In [ ]:
# Block-matrix cosine similarity → top-M = 20 per movie
TOP_M = 20; BLOCK = 2048
n = len(ids)
similar = {}   # movieId -> list[(sim_movieId, score, rank)]
for start in range(0, n, BLOCK):
    end = min(start + BLOCK, n)
    sims_block = Xn[start:end] @ Xn.T          # (block, 87585) float32 ~ 717MB max
    # bỏ self + no-genres (sim luôn 0) tự nhiên qua score
    for bi in range(end - start):
        gi = start + bi
        sims_block[bi, gi] = -1.0
        idx = np.argpartition(sims_block[bi], -TOP_M)[-TOP_M:]
        idx = idx[np.argsort(-sims_block[bi][idx])]
        row = [(int(ids[j]), round(float(sims_block[bi][j]), 4), k + 1)
               for k, j in enumerate(idx) if sims_block[bi][j] > 0]
        if row: similar[int(ids[gi])] = row[:TOP_M]
    del sims_block
print(f'movies with non-empty similar list: {len(similar):,} / {n:,}')
no_sim = n - len(similar)
print(f'movies with no genre overlap (empty similar): {no_sim:,} — expected: (no genres listed) movies = 7,080 (EDA §2)')

In [ ]:
# GATE (PLAN B3.3): lookup hợp lệ + contract §3.2 + save artifact
# 1) Sample lookup: movie 296 (Pulp Fiction) phải có similar hợp lệ
assert 296 in similar and len(similar[296]) == TOP_M, 'KILL: lookup Pulp Fiction không đủ TOP_M'
print('sample movieId=296 top5:', similar[296][:5])

# 2) No self-reference, score ∈ (0,1], rank 1..M, sorted desc
import random
for mid in random.sample(list(similar), 200):
    items = similar[mid]
    assert all(s['rank'] == i + 1 for i, s in enumerate(  # rank check
            [{'rank': r, 'movieId': m, 'score': s} for m, s, r in items]))
    assert all(m != mid for m, s, r in items), 'self-reference'
    assert all(0 < s <= 1 for m, s, r in items), 'score out of (0,1]'
print('contract sample checks: 200 movies PASS')

# 3) Save full artifact (chunked JSON per contract schema)
import json, datetime
version, gen_at = 'v1.0.0', datetime.datetime.utcnow().isoformat() + 'Z'
with open(f'{ARTIFACTS}/similar_movies.json', 'w') as f:
    docs = [{'movieId': mid, 'modelVersion': version, 'generatedAt': gen_at,
             'similar': [{'movieId': m, 'score': s, 'rank': r} for m, s, r in items]}
            for mid, items in similar.items()]
    json.dump(docs, f)
pd.DataFrame({'n_movies': [n], 'n_with_similar': [len(similar)],
              'n_no_genre_overlap': [no_sim], 'top_m': [TOP_M],
              'method': ['genres_multihot_cosine_blockmatrix']}).to_csv(f'{EVID}/content_based_stats.csv', index=False)
print('similar_movies.json + content_based_stats.csv saved')

## ✅ Notebook 02 DONE khi:
- Pulp Fiction lookup đủ 20 similar (đã assert)
- 200 movies ngẫu nhiên pass contract checks (đã assert)
- Số phim empty similar ≈ 7,080 (no-genres) — đối chiếu EDA §2
- Copy `similar_movies.json` + stats về repo, cập nhật CHECKLIST + WORKLOG

**Known limitation (ghi vào MODEL_DESIGN):** genres-only similarity overspecialize
(PRD risk: novelty thấp) → optional fusion Popularity ở serving (Person 2).